# M1-B1 — Exploration de l'historique des réservations

Ce notebook sert aux tâches 1 à 3 : connexion à la base, exploration en SQL, chargement des données.
Crée d'autres notebooks pour la suite (par exemple `02_modelisation.ipynb`).

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # pour pouvoir importer model.train_baseline plus tard
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

## Tâche 1 — La base répond-elle ?

In [2]:
pd.read_sql(text("SELECT COUNT(*) AS nb_reservations FROM reservations"), engine)

,nb_reservations
0,78703


## Tâche 2 — Explorer en SQL

Les calculs se font **dans la requête SQL** ; pandas sert seulement à afficher le résultat.

Pour commencer : les tables disponibles et leurs colonnes.

In [3]:
pd.read_sql(text("SELECT * FROM hotels"), engine)

,hotel_id,code,nom,type,ville
0,1,City Hotel,Escale Atlantique Centre,urbain,Lisbonne
1,2,Resort Hotel,Escale Atlantique Océan,resort,Algarve


In [4]:
pd.read_sql(text("SELECT * FROM reservations LIMIT 5"), engine)

,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,1,1,0,6,2015,July,27,1,0,2,...,6,None,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01
1,2,1,1,88,2015,July,27,1,0,4,...,9,None,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01
2,3,1,1,65,2015,July,27,1,0,4,...,9,None,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01
3,4,1,1,92,2015,July,27,1,2,4,...,9,None,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01
4,5,1,1,257,2015,July,27,1,0,2,...,6,None,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01


### Réservations et taux d'annulation par hôtel

In [5]:
query = text("""
    SELECT h.nom,
           h.code,
           COUNT(*)                            AS nb_reservations,
           ROUND(100 * AVG(r.is_canceled), 1)  AS taux_annulation_pct
    FROM reservations AS r
    JOIN hotels AS h ON h.hotel_id = r.hotel_id
    GROUP BY h.nom, h.code
    ORDER BY h.nom
""")
pd.read_sql(query, engine)

,nom,code,nb_reservations,taux_annulation_pct
0,Escale Atlantique Centre,City Hotel,51822,41.3
1,Escale Atlantique Océan,Resort Hotel,26881,26.3


### Taux d'annulation selon le mois d'arrivée

In [6]:
query = text("""
    SELECT to_char(date_trunc('month', arrival_date), 'YYYY-MM') AS mois_arrivee,
           COUNT(*)                                             AS nb_reservations,
           ROUND(100 * AVG(is_canceled), 1)                     AS taux_annulation_pct
    FROM reservations
    GROUP BY mois_arrivee
    ORDER BY mois_arrivee
""")
pd.read_sql(query, engine)

,mois_arrivee,nb_reservations,taux_annulation_pct
0,2015-07,2776,45.4
1,2015-08,3889,41.1
2,2015-09,5114,40.9
3,2015-10,4957,34.9
4,2015-11,2340,20.8
5,2015-12,2920,33.3
6,2016-01,2248,24.8
7,2016-02,3891,34.4
8,2016-03,4824,30.6
9,2016-04,5428,38.0


### Taux d'annulation selon le délai entre réservation et arrivée

In [7]:
query = text("""
    SELECT CASE
               WHEN lead_time <= 7   THEN '1. 0 à 7 jours'
               WHEN lead_time <= 30  THEN '2. 8 à 30 jours'
               WHEN lead_time <= 90  THEN '3. 31 à 90 jours'
               WHEN lead_time <= 180 THEN '4. 91 à 180 jours'
               WHEN lead_time <= 365 THEN '5. 181 à 365 jours'
               ELSE '6. plus d''un an'
           END                               AS delai_reservation,
           COUNT(*)                          AS nb_reservations,
           ROUND(100 * AVG(is_canceled), 1)  AS taux_annulation_pct
    FROM reservations
    GROUP BY delai_reservation
    ORDER BY delai_reservation
""")
pd.read_sql(query, engine)

,delai_reservation,nb_reservations,taux_annulation_pct
0,1. 0 à 7 jours,13925,10.0
1,2. 8 à 30 jours,12602,26.5
2,3. 31 à 90 jours,20140,35.5
3,4. 91 à 180 jours,15748,43.4
4,5. 181 à 365 jours,14504,57.7
5,6. plus d'un an,1784,77.6


### Taux d'annulation par segment de clientèle et par type d'acompte

In [8]:
query = text("""
    SELECT market_segment,
           COUNT(*)                          AS nb_reservations,
           ROUND(100 * AVG(is_canceled), 1)  AS taux_annulation_pct
    FROM reservations
    GROUP BY market_segment
    ORDER BY nb_reservations DESC
""")
pd.read_sql(query, engine)

,market_segment,nb_reservations,taux_annulation_pct
0,Online TA,33826,33.9
1,Offline TA/TO,18552,34.5
2,Groups,13957,62.3
3,Direct,7977,15.3
4,Corporate,3733,15.9
5,Complementary,529,11.7
6,Aviation,127,22.8
7,Undefined,2,100.0


In [9]:
query = text("""
    SELECT deposit_type,
           COUNT(*)                          AS nb_reservations,
           ROUND(100 * AVG(is_canceled), 1)  AS taux_annulation_pct,
           ROUND(AVG(lead_time))             AS delai_moyen_jours
    FROM reservations
    GROUP BY deposit_type
    ORDER BY nb_reservations DESC
""")
pd.read_sql(query, engine)

,deposit_type,nb_reservations,taux_annulation_pct,delai_moyen_jours
0,No Deposit,68107,26.6,82.0
1,Non Refund,10461,99.2,226.0
2,Refundable,135,7.4,155.0


### Valeurs manquantes ou incohérentes

In [10]:
query = text("""
    SELECT COUNT(*) FILTER (WHERE children IS NULL)  AS children_vides,
           COUNT(*) FILTER (WHERE country IS NULL)   AS country_vides,
           COUNT(*) FILTER (WHERE agent IS NULL)     AS agent_vides,
           COUNT(*) FILTER (WHERE company IS NULL)   AS company_vides,
           COUNT(*) FILTER (WHERE adults + COALESCE(children, 0) + babies = 0)        AS sans_voyageur,
           COUNT(*) FILTER (WHERE stays_in_weekend_nights + stays_in_week_nights = 0) AS sans_nuit
    FROM reservations
""")
pd.read_sql(query, engine)

,children_vides,country_vides,agent_vides,company_vides,sans_voyageur,sans_nuit
0,4,405,10676,74216,113,571


### Rapport d'étonnement

- **L'acompte non remboursable annule presque toujours** : 99,2 % d'annulation sur 10 461 réservations `Non Refund`, contre 26,6 % sans acompte. C'est l'inverse de l'intuition (on n'abandonne pas un acompte perdu). Ces réservations sont prises très tôt (226 jours en moyenne) : probablement des blocs d'agences ou de groupes, saisis puis annulés. Le modèle s'appuiera fortement dessus.
- **Plus on réserve tôt, plus on annule** : 10 % d'annulation à moins de 8 jours, 57,7 % entre 6 mois et 1 an, 77,6 % au-delà d'un an.
- **Les groupes annulent 4 fois plus que la vente directe** : 62,3 % pour `Groups`, 15,3 % pour `Direct`, 15,9 % pour `Corporate`. Les agences (en ligne ou non) sont autour de 34 %.
- **L'hôtel de ville annule plus que le resort** : 41,3 % à Escale Atlantique Centre, 26,3 % à Escale Atlantique Océan.
- **Le taux change beaucoup d'un mois à l'autre**, de 20,8 % (novembre 2015) à 45,4 % (juillet 2015), sans saisonnalité régulière : juillet passe de 45,4 % en 2015 à 32,8 % en 2016. Un modèle entraîné sur le passé verra donc un futur un peu différent, ce qu'un découpage aléatoire cacherait.
- **Globalement, plus d'une réservation sur trois est annulée** (28 479 sur 78 703, soit 36,2 %).
- **Des trous à traiter** : `company` est vide 94 fois sur 100 et `agent` 14 fois sur 100 (réservation sans intermédiaire, ce n'est pas une erreur), `country` 405 fois, `children` 4 fois. 113 réservations n'ont aucun voyageur et 571 aucune nuit.

## Tâche 3 — Charger les données et préparer le découpage

Une seule requête SQL, avec la jointure sur `hotels` pour récupérer la colonne `hotel`
(`'City Hotel'` / `'Resort Hotel'`) attendue par la recette de la v1.

In [ ]:
# TODO 

In [ ]:
# TODO 